# Phishing Detection Agent

## Notebook Setup

Install dependencies before running the cells:

```bash
pip install -r requirements.txt
```

### Required Libraries
- pandas
- numpy
- scikit-learn
- xgboost
- sentence-transformers
- matplotlib
- pyarrow
- scipy

#### 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import hstack
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_sample_weight
import pickle
import matplotlib.pyplot as plt

#### 2. Data Loading, Preprocessing, and Visualization

This cell prepares our raw data so the machine learning model can learn from it effectively. It performs the following critical steps:

1. **Loading Modern Data:** Reads the high-fidelity Hugging Face dataset (`.parquet` format) for modern email threat intelligence, alongside our existing CSV datasets for SMS messages.
2. **Filtering & Standardization:** Filters out raw URL data to focus strictly on text-based NLP processing, renames columns to ensure consistency across all sources (mapping everything to `label` and `text_combined`), and fuses the diverse datasets into one unified pipeline.
3. **The 4-Class Enterprise Matrix:** Unifies the classifications into a streamlined, threat-focused 4-class system. We intentionally drop the ambiguous "Spam Email" (nuisance) class to force the model to focus strictly on actual cybersecurity threats:
   - `0` -> Safe Email
   - `1` -> Safe SMS
   - `2` -> Phishing Email
   - `3` -> Malicious SMS
4. **Visualization:** Creates a bar chart to visualize the distribution of messages across these four categories. This allows us to audit the balance of our training data (such as the 50/50 split in emails) and prepare for mathematical class-weight adjustments on the SMS data.

In [ ]:
print("Loading dataset...")
df_phishing = pd.read_parquet('datasets/phishing_email.parquet')
df_phishing_sms = pd.read_csv('datasets/phishing_sms.csv')
df_spam_sms = pd.read_csv('datasets/spam_sms.csv')

# Process phishing email dataset
df_phishing_email= df_phishing[df_phishing['label'].isin([0, 1])].copy()

# rename columns to ensure consistency
df_phishing_email = df_phishing_email.rename(columns={'label': 'label', 'content': 'text_combined'})
df_phishing_sms = df_phishing_sms.rename(columns={'LABEL': 'label', 'TEXT': 'text_combined'})
df_spam_sms = df_spam_sms.rename(columns={'v1': 'label', 'v2': 'text_combined'})

# replace label values based on classification scheme
print("Mapping labels to unified classification scheme...")
df_phishing_email['label'] = df_phishing_email['label'].map({0: 0, 1: 2})
df_phishing_sms['label'] = df_phishing_sms['label'].map({'ham': 1, 'Smishing': 3})
df_spam_sms['label'] = df_spam_sms['label'].map({'ham': 1, 'spam': 3})

# Combine datasets
dataset = pd.concat([df_phishing_email, df_phishing_sms, df_spam_sms], ignore_index=True)

# Ensure the label column is correctly cast to integers
dataset['label'] = pd.to_numeric(dataset['label'], errors='coerce')
dataset = dataset.dropna()
dataset['label'] = dataset['label'].astype(int)

display(dataset.head())

# Map integer labels to their string representations for the plot
label_map = {
    0: 'Safe Email',
    1: 'Safe SMS',
    2: 'Phishing Email',
    3: 'Malicious SMS'
}

dataset['label_name'] = dataset['label'].map(label_map)

plt.figure(figsize=(10, 6))
# Calculate counts and sort by the original label index to maintain a logical order
counts = dataset['label_name'].value_counts()

category_colors = {
    'Safe Email': '#4CAF50',     # Green
    'Safe SMS': '#66BB6A',       # Light Green
    'Phishing Email': '#F44336', # Red
    'Malicious SMS': '#EF5350'    # Light Red
}
bar_colors = [category_colors[label] for label in counts.index]
ax = counts.plot(kind='bar', color=bar_colors, edgecolor='black')

plt.title('Distribution of Messages by Category', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, fontsize=12)
plt.xlabel('Message Category', fontsize=12, fontweight='bold')
plt.ylabel('Number of Messages', fontsize=12, fontweight='bold')

# Add data labels on top of the bars
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 8),
                textcoords='offset points',
                fontsize=11, fontweight='bold')

# Clean up aesthetics
plt.grid(axis='y', linestyle='--', alpha=0.7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

#### 4. Hybrid Feature Engineering, Embedding, and Data Splitting

This cell is the core of our **Hybrid Architecture**. It extracts both the structural logic and the semantic meaning of the messages before splitting the data for training. It performs four critical steps:

1. **Structural Feature Extraction & Cleaning:** It uses Regular Expressions (Regex) to explicitly identify structural threat vectors (e.g., the presence of a URL, message length, and urgency keywords). Crucially, it cleans the text by replacing messy links with a `[URL]` placeholder so the transformer doesn't get blinded by visual noise.
2. **Semantic Text Embedding:** It uses a pre-trained NLP transformer (`all-MiniLM-L6-v2`) to read the *cleaned* text and convert the psychological intent of the message into dense numerical vectors (384 dimensions).
3. **Matrix Fusion:** It horizontally stacks the 384 semantic vectors with our 3 structural features (`np.hstack`). This creates a unified 387-dimensional "super-matrix" so the XGBoost model can judge both the grammar and the structure of the message simultaneously.
4. **Train-Test Split:** It splits the fused dataset into an 80% chunk for training the model (`x_train`, `y_train`) and a 20% chunk for testing (`x_test`, `y_test`). The `stratify=y` parameter ensures that the exact distribution of our 4-Class Threat Matrix is perfectly maintained across both sets.

In [ ]:
print("Engineering Structural Features and Cleaning Text...")

url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+|www\.[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

# 1. Extract the URL feature
dataset['contains_url'] = dataset['text_combined'].apply(lambda x: 1 if re.search(url_pattern, str(x)) else 0)

# 2. Clean the text by replacing the URL with a placeholder tag
dataset['text_clean'] = dataset['text_combined'].apply(lambda x: re.sub(url_pattern, ' [URL] ', str(x)))

# 3. Extract length
dataset['msg_length'] = dataset['text_combined'].apply(lambda x: len(str(x)))

# 4. Extract urgency
urgency_keywords = ['urgent', 'suspend', 'decline', 'hold', 'alert', 'verify', 'update', 'immediate', 'restricted', 'validate', 'restore']
urgency_pattern = '|'.join([rf'\b{word}\b' for word in urgency_keywords])
dataset['is_urgent'] = dataset['text_combined'].apply(lambda x: 1 if re.search(urgency_pattern, str(x), re.IGNORECASE) else 0)

print("Features extracted. Generating Dense Semantic Vectors on CLEANED text...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Generate vectors using the CLEANED text
semantic_vectors = embedder.encode(dataset['text_clean'].tolist(), show_progress_bar=False)

print("Fusing Semantic Vectors with Structural Features...")
structural_features = dataset[['contains_url', 'msg_length', 'is_urgent']].values
x_fused = np.hstack((semantic_vectors, structural_features))
y = dataset['label'].values

print(f"Final Matrix Shape: {x_fused.shape}")

x_train, x_test, y_train, y_test = train_test_split(x_fused, y, test_size=0.2, random_state=42, stratify=y)

#### 5. Training the Model and Evaluation

This cell trains the machine learning model and checks how well it performs:

1. **Handling Imbalance:** Calculates "sample weights" so the model pays equal attention to all categories, preventing it from ignoring rare message types (like malicious SMS).
2. **Training:** Configures and teaches an XGBoost classifier (a powerful, fast algorithm) using our 80% training data.
3. **Evaluation:** Tests the trained model against the 20% test data to calculate its overall accuracy score.
4. **Visualization:** Generates a detailed report and a "Confusion Matrix" chart to visualize exactly where the model predictions are correct and where it might be confusing certain message categories.

In [ ]:
# 1. Calculate sample weights for the imbalanced dataset
print("Calculating class weights...")
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# 2. Initialize Model
print("Training Hybrid XGBoost Model...")
model = XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    tree_method='hist',
    random_state=42
)

# 3. Fit Model on the fused data
model.fit(x_train, y_train, sample_weight=sample_weights)

# 4. Predict
print("Evaluating Hybrid Model...")
predictions = model.predict(x_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\nOverall Hybrid Model Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Report:")

# 5. Define labels
target_names = [
    'Safe Email (0)',
    'Safe SMS (1)',
    'Phishing Email (2)',
    'Malicious SMS (3)'
]
print(classification_report(y_test, predictions, target_names=target_names))

# 6. Visualize
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay.from_estimator(
    model, x_test, y_test,
    display_labels=target_names,
    cmap='Blues', values_format='d',
    xticks_rotation=45
)
plt.title('Hybrid 4-Class Threat Matrix - Confusion Matrix')
plt.tight_layout()
plt.show()

#### 6. Saving the model and vectorizer

In [ ]:
print("Saving XGBoost model architecture...")
path = 'threat_model_xgboost.pkl'

# save the XGBoost model.
with open(path, 'wb') as f:
    pickle.dump(model, f)

print("Model saved successfully as 'threat_model_xgboost.pkl'.")